# DRL Intro: MDP, Return, V/Q/Advantage – Monte-Carlo Evaluation (FrozenLake)

Ziel:
1) Episode sammeln (Rollout)
2) Returns G_t berechnen
3) Monte-Carlo Schätzung von V(s) und Q(s,a)
4) Advantage A(s,a) berechnen
5) Aus Q eine ε-greedy Policy ableiten
6) Zeigen, dass sich V(start) verbessert (Policy Improvement)


In [ ]:
!pip -q install gymnasium

import gymnasium as gym
import numpy as np
from collections import defaultdict

SEED = 42
rng = np.random.default_rng(SEED)


In [ ]:
# Environment: deterministisch für klare Ergebnisse
env = gym.make("FrozenLake-v1", is_slippery=False)

s0, info = env.reset(seed=SEED)
nS = env.observation_space.n
nA = env.action_space.n

print("nS:", nS, "nA:", nA, "start:", s0)


nS: 16 nA: 4 start: 0


## Policy und Rollout

- Policy: Funktion, die aus Zustand s eine Aktion a wählt.
- Rollout: wir lassen Agent+Env laufen und speichern (s, a, r) pro Schritt.


In [ ]:
def random_policy(s, nA):
    return int(rng.integers(nA))

def rollout_episode(env, policy_fn, max_steps=200, seed=0):
    traj = []  # list of (s, a, r)
    s, _ = env.reset(seed=seed)
    for _ in range(max_steps):
        a = policy_fn(s, env.action_space.n)
        s2, r, terminated, truncated, _ = env.step(a)
        traj.append((s, a, r))
        s = s2
        if terminated or truncated:
            break
    return traj

traj = rollout_episode(env, random_policy, seed=SEED)
print("episode length:", len(traj))
print("first steps:", traj[:8])


episode length: 4
first steps: [(0, 0, 0), (0, 3, 0), (0, 2, 0), (1, 1, 0)]


## Returns berechnen

Return G_t ist die discounted Summe der zukünftigen Rewards ab Schritt t.
Wir berechnen das rückwärts:
G = 0
G <- r + gamma * G


In [ ]:
def compute_returns(traj, gamma=0.99):
    G = 0.0
    returns = []
    # HA - Warum reversed?
    for (s, a, r) in reversed(traj):
        G = r + gamma * G
        returns.append(G)
    returns.reverse()
    return returns

gamma = 0.99
Gs = compute_returns(traj, gamma=gamma)
list(zip(traj[:8], Gs[:8]))


[((0, 0, 0), 0.0), ((0, 3, 0), 0.0), ((0, 2, 0), 0.0), ((1, 1, 0), 0.0)]

## MC Evaluation von V(s)

First-Visit MC:
- pro Episode zählt nur das erste Auftreten eines Zustands s
- V(s) = Durchschnitt der beobachteten Returns in s

```
seen = set()
for t, (s, a, r) in enumerate(traj):
    if s in seen:
        continue
    seen.add(s)
    V[s] += G[t]
```

Every-Visit MC:
- pro Episode zählt jedes Auftreten eines Zustands s
- V(s) ist der Durchschnitt der beobachteten Return über alle Besuche von s in allen Episoden.
```
for t, (s, a, r) in enumerate(traj):
    V[s] += G[t]
```


In [ ]:
def mc_evaluate_V(env, policy_fn, episodes=3000, gamma=0.99, seed=0):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)

    for ep in range(episodes):
        traj = rollout_episode(env, policy_fn, seed=seed + ep)
        Gs = compute_returns(traj, gamma=gamma)

        seen = set()
        for t, (s, a, r) in enumerate(traj):
            if s in seen:
                continue
            seen.add(s)
            returns_sum[s] += Gs[t]
            returns_count[s] += 1

    V = {s: returns_sum[s] / returns_count[s] for s in returns_count}
    return V

V_rand = mc_evaluate_V(env, random_policy, episodes=4000, gamma=gamma, seed=SEED)

s0, _ = env.reset(seed=SEED)
print("V_random(start):", round(V_rand.get(s0, 0.0), 4))


V_random(start): 0.0124


In [ ]:
V_rand

{0: 0.01243207145196423,
 4: 0.01748787153372368,
 1: 0.007234537243722057,
 2: 0.012189950638019269,
 3: 0.0020033272971132473,
 6: 0.026821612870081044,
 8: 0.03845824015901537,
 9: 0.08792815679695472,
 10: 0.12013856417409896,
 13: 0.18086616947046652,
 14: 0.40231443517025645}

## MC Evaluation von Q(s,a)

Analog:
- wir mitteln Returns pro (s,a)
- daraus können wir greedy / ε-greedy Policies bauen


In [ ]:
def mc_evaluate_Q(env, policy_fn, episodes=6000, gamma=0.99, seed=0):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)

    for ep in range(episodes):
        traj = rollout_episode(env, policy_fn, seed=seed + ep)
        Gs = compute_returns(traj, gamma=gamma)

        seen_sa = set()
        for t, (s, a, r) in enumerate(traj):
            key = (s, a)
            if key in seen_sa:
                continue
            seen_sa.add(key)
            returns_sum[key] += Gs[t]
            returns_count[key] += 1

    Q = {k: returns_sum[k] / returns_count[k] for k in returns_count}
    return Q

Q_rand = mc_evaluate_Q(env, random_policy, episodes=8000, gamma=gamma, seed=SEED)
print("Q entries:", list(Q_rand.items())[:5])


Q entries: [((0, 1), 0.014838457167547595), ((4, 0), 0.017789925010889333), ((4, 3), 0.01565071370145109), ((4, 2), 0.0), ((0, 2), 0.010242417311593928)]


## Advantage A(s,a)

A(s,a) = Q(s,a) - V(s)
Interpretation: wie viel besser/schlechter ist Aktion a gegenüber dem "Durchschnitt" in s.


In [ ]:
def advantage(V, Q, s, a):
    return Q.get((s, a), 0.0) - V.get(s, 0.0)

# Advantage im Startzustand für alle Aktionen
adv_start = [(a, advantage(V_rand, Q_rand, s0, a)) for a in range(nA)]
adv_start


[(0, 3.3547907898329524e-05),
 (1, 0.0024063857155833656),
 (2, -0.0021896541403703014),
 (3, -0.0002846759044804973)]

## Policy Improvement: ε-greedy aus Q

- greedy: a = argmax_a Q(s,a)
- ε-greedy: mit Wahrscheinlichkeit ε zufällig, sonst greedy

Dann evaluieren wir die neue Policy wieder mit MC und vergleichen V(start).


In [ ]:
def epsilon_greedy_policy_from_Q(Q, nA, eps=0.1):
    def policy(s, nA_ignored=None):
        if rng.random() < eps:
            return int(rng.integers(nA))
        qs = [Q.get((s, a), 0.0) for a in range(nA)]
        return int(np.argmax(qs))
    return policy

pi_eps = epsilon_greedy_policy_from_Q(Q_rand, nA, eps=0.1)

V_eps = mc_evaluate_V(env, pi_eps, episodes=4000, gamma=gamma, seed=SEED)
print("V_random(start):", round(V_rand.get(s0, 0.0), 4))
print("V_eps(start):   ", round(V_eps.get(s0, 0.0), 4))


V_random(start): 0.0124
V_eps(start):    0.8472


## Mini Loop: wiederholte Verbesserung (Iteration)

Wir wiederholen:
1) Q unter aktueller Policy schätzen
2) neue ε-greedy Policy bauen
3) V(start) loggen

Achtung: Das ist noch nicht "Policy Iteration" im strengen Sinn,
aber zeigt sehr gut die Grundidee: Bessere Wertschätzungen (V/Q) → bessere Entscheidungsgrundlage → verbesserte Policy


In [ ]:
def policy_improvement_loop(env, init_policy, iters=5, eps=0.1, episodes_Q=6000, episodes_V=3000, gamma=0.99, seed=0):
    policy = init_policy
    history = []

    s0, _ = env.reset(seed=seed)

    for k in range(iters):
        Q = mc_evaluate_Q(env, policy, episodes=episodes_Q, gamma=gamma, seed=seed + 1000*k)
        policy = epsilon_greedy_policy_from_Q(Q, env.action_space.n, eps=eps)
        V = mc_evaluate_V(env, policy, episodes=episodes_V, gamma=gamma, seed=seed + 2000*k)
        history.append((k, V.get(s0, 0.0)))

    return history

hist = policy_improvement_loop(env, random_policy, iters=6, eps=0.1, episodes_Q=6000, episodes_V=3000, gamma=gamma, seed=SEED)
hist


## Was haben wir heute gelernt?

- Reward vs Return: Return ist das Ziel, nicht der einzelne Reward.
- V(s) und Q(s,a) sind Erwartungswerte von Returns.
- Monte-Carlo schätzt diese Werte aus Episoden (ohne Modell von P).
- Advantage erklärt "wie gut ist diese Aktion relativ zum Durchschnitt in s".
- Aus Q kann man eine bessere Policy ableiten (ε-greedy).


## Policy Improvement - Wie lernt der Agent?

In [ ]:
def epsilon_linear(k, eps0=0.3, eps_min=0.02, decay_steps=10):
    # fällt linear ab von eps0 zu eps_min über decay_steps Iterationen
    eps = eps0 - (eps0 - eps_min) * (k / decay_steps)
    return float(max(eps_min, eps))

def epsilon_exp(k, eps0=0.3, eps_min=0.02, alpha=0.85):
    eps = eps0 * (alpha ** k)
    return float(max(eps_min, eps))


In [ ]:
def greedy_action_from_Q(Q, s, nA):
    qs = [Q.get((s, a), 0.0) for a in range(nA)]
    return int(np.argmax(qs))

def make_epsilon_greedy_policy(Q, nA, eps):
    def policy(s, nA_ignored=None):
        if rng.random() < eps:
            return int(rng.integers(nA))
        return greedy_action_from_Q(Q, s, nA)
    return policy


In [ ]:
def evaluate_success_rate(env, policy_fn, episodes=1000, seed=0, max_steps=200):
    successes = 0
    for ep in range(episodes):
        s, _ = env.reset(seed=seed + ep)
        total_reward = 0
        for _ in range(max_steps):
            a = policy_fn(s, env.action_space.n)
            s, r, terminated, truncated, _ = env.step(a)
            total_reward += r
            if terminated or truncated:
                break
        if total_reward > 0:
            successes += 1
    return successes / episodes


In [ ]:
def policy_improvement_loop_mc_control_decay(
    env,
    init_policy,
    iters=8,
    eps_schedule="linear",      # "linear" oder "exp"
    eps0=0.3,
    eps_min=0.02,
    decay_steps=10,             # für linear
    alpha=0.85,                 # für exp
    episodes_Q=6000,
    episodes_V=3000,
    eval_episodes=1000,
    gamma=0.99,
    seed=0
):
    policy = init_policy
    history = []
    s0, _ = env.reset(seed=seed)

    for k in range(iters):
        # 1) Evaluate current policy -> estimate Q
        Q = mc_evaluate_Q(env, policy, episodes=episodes_Q, gamma=gamma, seed=seed + 1000*k)

        # 2) Choose epsilon for this iteration
        if eps_schedule == "linear":
            eps = epsilon_linear(k, eps0=eps0, eps_min=eps_min, decay_steps=decay_steps)
        elif eps_schedule == "exp":
            eps = epsilon_exp(k, eps0=eps0, eps_min=eps_min, alpha=alpha)
        else:
            raise ValueError("eps_schedule must be 'linear' or 'exp'")

        # 3) Improve policy using greedy(max) with epsilon exploration
        policy = make_epsilon_greedy_policy(Q, env.action_space.n, eps=eps)

        # 4) Track progress
        V = mc_evaluate_V(env, policy, episodes=episodes_V, gamma=gamma, seed=seed + 2000*k)
        v0 = float(V.get(s0, 0.0))
        sr = evaluate_success_rate(env, policy, episodes=eval_episodes, seed=seed + 3000*k)

        history.append({"iter": k, "eps": eps, "V(start)": v0, "success_rate": sr})
        print(f"iter={k:02d}  eps={eps:.3f}  V(start)={v0:.4f}  success_rate={sr:.3f}")

    return policy, Q, history


In [ ]:
trained_epsilon_greedy_policy_func, q_final, hist = policy_improvement_loop_mc_control_decay(env, random_policy, seed=SEED)

iter=00  eps=0.300  V(start)=0.5990  success_rate=0.635
iter=01  eps=0.272  V(start)=0.3515  success_rate=0.373
iter=02  eps=0.244  V(start)=0.6902  success_rate=0.737
iter=03  eps=0.216  V(start)=0.5331  success_rate=0.677
iter=04  eps=0.188  V(start)=0.3519  success_rate=0.427
iter=05  eps=0.160  V(start)=0.7580  success_rate=0.815
iter=06  eps=0.132  V(start)=0.8155  success_rate=0.862
iter=07  eps=0.104  V(start)=0.3171  success_rate=0.463


In [ ]:
q_final

{(0, 2): 0.8159542679868008,
 (1, 2): 0.8520639947307208,
 (2, 1): 0.864632393221832,
 (6, 1): 0.9391627430187036,
 (10, 1): 0.9857155513344252,
 (14, 2): 1.0,
 (2, 3): 0.8853381217014685,
 (14, 0): 0.932614667436372,
 (13, 2): 0.9824536820809254,
 (1, 0): 0.842320524557883,
 (6, 0): 0.0,
 (0, 3): 0.8408814601347393,
 (10, 2): 0.0,
 (10, 3): 0.8585475160733005,
 (14, 3): 0.9456025362391429,
 (10, 0): 0.930418384006909,
 (9, 2): 0.9440000429351596,
 (6, 3): 0.826775367690673,
 (2, 0): 0.844187593847408,
 (14, 1): 0.9828709371047382,
 (0, 1): 0.8212242992344381,
 (4, 1): 0.8476812723371214,
 (8, 2): 0.8963426369564509,
 (0, 0): 0.830680825759631,
 (1, 1): 0.0,
 (2, 2): 0.8133796772560579,
 (3, 0): 0.8530676635317401,
 (4, 3): 0.8069829852008572,
 (6, 2): 0.0,
 (1, 3): 0.8288372981604216,
 (3, 2): 0.9471955996008988,
 (3, 3): 0.9509900498999999,
 (9, 0): 0.9547894211189943,
 (4, 2): 0.0,
 (13, 0): 0.0,
 (13, 3): 0.9647821571142856,
 (9, 3): 0.0,
 (3, 1): 0.0,
 (9, 1): 0.9761894009999998,


In [ ]:

greedy_trained_policy = make_epsilon_greedy_policy(q_final, env.action_space.n, eps=0.0)

In [ ]:
_, q_evolution, hist = policy_improvement_loop_mc_control_decay(env, greedy_trained_policy, seed=SEED)

iter=00  eps=0.300  V(start)=0.0000  success_rate=0.000
iter=01  eps=0.272  V(start)=0.5133  success_rate=0.608
iter=02  eps=0.244  V(start)=0.6957  success_rate=0.724
iter=03  eps=0.216  V(start)=0.7159  success_rate=0.763
iter=04  eps=0.188  V(start)=0.7385  success_rate=0.763
iter=05  eps=0.160  V(start)=0.7741  success_rate=0.843
iter=06  eps=0.132  V(start)=0.4548  success_rate=0.619
iter=07  eps=0.104  V(start)=0.8532  success_rate=0.887
